# Лабораторная работа № 3. Статистический анализ продаж с помощью NumPy
**Кейс:** региональная эффективность интернет-магазина (продажи в тыс. ден. ед. за 8 недель).

Ограничения соблюдены: в решении нет циклов `for`/`while`, нет Pandas и ручного перебора элементов — только векторизованные операции NumPy, булевы маски и функции агрегации.
Строки массива `sales` — регионы, столбцы — недели, поэтому `axis=1` агрегирует по неделям (итог по региону), `axis=0` — по регионам (итог по неделе).

In [1]:
import sys
import numpy as np

np.set_printoptions(suppress=True, precision=2, linewidth=140)

sales = np.array([
    [120, 125, 130, 128, 135, 140, 138, 145],
    [80, 82, 79, 85, 88, 90, 86, 92],
    [150, 148, 155, 160, 158, 165, 220, 170],
    [60, 65, 62, 58, 64, 63, 61, 66],
    [100, 105, 102, 108, 110, 112, 109, 115]
])
region_names = np.array(["Bishkek", "Osh", "Karakol", "Tokmok", "Jalal-Abad"])

def table(columns, dec=2, first_width=12, width=13):
    """Печать таблицы без циклов: первый столбец — названия, остальные — числа."""
    names = np.array(list(columns.keys()))
    arrays = list(columns.values())
    rec = np.rec.fromarrays(arrays, names=[f"c{i}" for i in range(len(arrays))])
    fmt = [f"%-{first_width}s"] + [f"%{width}.{dec}f"] * (len(arrays) - 1)
    head = " ".join([f"{names[0]:<{first_width}}"] + list(np.char.rjust(names[1:], width)))
    np.savetxt(sys.stdout, rec, fmt=" ".join(fmt), header=head, comments="")

# Проверки входных данных
print("Форма массива (регионы x недели):", sales.shape)
print("Пропусков/NaN:", np.isnan(sales.astype(float)).sum(), "| отрицательных значений:", (sales < 0).sum())
assert sales.shape == (region_names.size, 8)

Форма массива (регионы x недели): (5, 8)
Пропусков/NaN: 0 | отрицательных значений: 0


## Задание 1. Сравнение регионов
Считаем суммарную выручку, среднее и медиану недельных продаж по каждому региону (`axis=1`).

In [2]:
total = sales.sum(axis=1)
mean = sales.mean(axis=1)
median = np.median(sales, axis=1)

table({"Регион": region_names, "Сумма": total, "Среднее": mean, "Медиана": median, "Среднее-Медиана": mean - median})

leader = region_names[np.argmax(total)]
print("\nЛидер по суммарным продажам:", leader, "-", total.max(), "тыс.")

# Проверка: сумма по регионам и по неделям даёт один и тот же общий итог
assert sales.sum(axis=0).sum() == total.sum()
print("Общая выручка сети:", total.sum(), "тыс.")

Регион               Сумма       Среднее       Медиана Среднее-Медиана
Bishkek            1061.00        132.62        132.50          0.12
Osh                 682.00         85.25         85.50         -0.25
Karakol            1326.00        165.75        159.00          6.75
Tokmok              499.00         62.38         62.50         -0.12
Jalal-Abad          861.00        107.62        108.50         -0.88

Лидер по суммарным продажам: Karakol - 1326 тыс.
Общая выручка сети: 4429 тыс.


**Выводы.**
- **Лидер по объёму — Karakol**: 1326 тыс. (почти 30% выручки сети), за ним Bishkek (1061) и Jalal-Abad (861). Замыкает список Tokmok (499).
- **Типичную недельную выручку** лучше описывает **медиана**: она не зависит от единичных всплесков. У четырёх регионов среднее и медиана почти совпадают (разница ≤ 1 тыс.), а у Karakol среднее (165,75) заметно выше медианы (159): одна аномальная неделя «тянет» среднее вверх на ~6,75 тыс. Поэтому для Karakol типичным значением следует считать **159 тыс. в неделю**, а не 165,75.
- Если понимать «наиболее типичную выручку» как значение, которому можно доверять, лучше всего это выполняется у **Tokmok** (медиана 62,5 при самом малом относительном разбросе IQR/медиана — см. задание 2).

## Задание 2. Анализ разброса
Абсолютное стандартное отклонение зависит от масштаба продаж, поэтому дополнительно считаем относительные меры:
- **коэффициент вариации** CV = σ / среднее (используем `ddof=1`, так как 8 недель — это выборка);
- **IQR / медиана** — робастный аналог, не чувствительный к выбросам.

In [3]:
std0 = sales.std(axis=1)            # ddof=0 (генеральная совокупность)
std1 = sales.std(axis=1, ddof=1)    # выборочное
q1 = np.percentile(sales, 25, axis=1)
q3 = np.percentile(sales, 75, axis=1)
iqr = q3 - q1

assert np.all(mean > 0) and np.all(median > 0)   # знаменатели не равны нулю
cv = std1 / mean * 100
iqr_rel = iqr / median * 100

table({"Регион": region_names, "std(ddof=0)": std0, "std(ddof=1)": std1, "IQR": iqr, "CV, %": cv, "IQR/медиана,%": iqr_rel})

rank_cv = np.argsort(cv)
print("\nОт самого стабильного к наименее стабильному (по CV):", region_names[rank_cv])

Регион         std(ddof=0)   std(ddof=1)           IQR         CV, % IQR/медиана,%
Bishkek               7.81          8.35         11.25          6.29          8.49
Osh                   4.38          4.68          7.00          5.49          8.19
Karakol              21.60         23.09         12.50         13.93          7.86
Tokmok                2.50          2.67          3.50          4.28          5.60
Jalal-Abad            4.72          5.04          6.25          4.68          5.76

От самого стабильного к наименее стабильному (по CV): ['Tokmok' 'Jalal-Abad' 'Osh' 'Bishkek' 'Karakol']


**Выводы.**
- По абсолютному σ «самый нестабильный» Karakol (23,1 против 2,7–8,3 у остальных), но это частично эффект масштаба и одного выброса.
- По **CV** картина такая: Tokmok (4,3%) и Jalal-Abad (4,7%) — самые стабильные; Osh (5,5%) и Bishkek (6,3%) — умеренно стабильны; **Karakol (13,9%) — явно нестабилен**, и CV почти втрое выше, чем у стабильных регионов.
- По **IQR/медиана** Karakol (7,9%) — на уровне Bishkek (8,5%) и Osh (8,2%). Значит, «ядро» продаж Karakol стабильно, а высокий σ и CV объясняются почти целиком одной неделей (неделя 7). Tokmok и Jalal-Abad остаются самыми стабильными (5,6–5,8%).
- Абсолютные значения σ разных регионов напрямую сравнивать нельзя: у Tokmok σ = 2,7 при среднем 62 (4,3%), а у Osh σ = 4,7 при среднем 85 (5,5%).

## Задание 3. Поиск выбросов (правило IQR)
Границы для каждого региона: `Q1 - 1.5*IQR` и `Q3 + 1.5*IQR`. Сравнение выполняется булевой маской по всей матрице сразу (границы приводятся к форме `(5, 1)` для броадкастинга).

In [4]:
lower = (q1 - 1.5 * iqr)[:, None]
upper = (q3 + 1.5 * iqr)[:, None]
outlier_mask = (sales < lower) | (sales > upper)

table({"Регион": region_names, "Нижняя гр.": lower.ravel(), "Верхняя гр.": upper.ravel(),
       "Мин": sales.min(axis=1), "Макс": sales.max(axis=1), "Выбросов": outlier_mask.sum(axis=1)}, dec=2)

r_idx, w_idx = np.nonzero(outlier_mask)
print("\nНайденные потенциальные выбросы:")
print("Регион:", region_names[r_idx], "| неделя:", w_idx + 1, "| значение:", sales[outlier_mask])
print("Медиана региона:", median[r_idx], "| отклонение от медианы, %:", (sales[outlier_mask] / median[r_idx] - 1) * 100)
print("Соседние недели (6 и 8):", sales[r_idx, w_idx - 1], sales[r_idx, w_idx + 1])

Регион          Нижняя гр.   Верхняя гр.           Мин          Макс      Выбросов
Bishkek             110.38        155.38        120.00        145.00          0.00
Osh                  71.00         99.00         79.00         92.00          0.00
Karakol             135.00        185.00        148.00        220.00          1.00
Tokmok               55.50         69.50         58.00         66.00          0.00
Jalal-Abad           94.88        119.88        100.00        115.00          0.00

Найденные потенциальные выбросы:
Регион: ['Karakol'] | неделя: [7] | значение: [220]
Медиана региона: [159.] | отклонение от медианы, %: [38.36]
Соседние недели (6 и 8): [165] [170]


**Результат:** найден один потенциальный выброс — **Karakol, неделя 7: 220 тыс.** при верхней границе 185 (медиана региона — 159, т.е. +38%; соседние недели — 165 и 170). В следующую неделю значение возвращается к 170 — всплеск разовый.

**Почему это может быть реальный всплеск:**
- распродажа, акция или сезонный пик (праздники, «чёрная пятница»);
- крупный разовый заказ (оптовый/корпоративный);
- запуск рекламной кампании или новый товар;
- возможна «отложенная» выручка: перенос части продаж из соседних недель.

**Почему это может быть ошибка данных:**
- опечатка при вводе (220 вместо 170 или 160 — характерный «залипший» символ);
- дублирование заказов или транзакций, ошибочная загрузка данных за две недели в одну;
- ошибочное включение возвратов/тестовых заказов, неверный курс или единицы измерения.

После пика значение возвращается к уровню тренда (170), что согласуется с разовым событием, но не исключает ошибку ввода. Только проверка первичных данных (чеки, выгрузки CRM) даёт окончательный ответ. Выход за границы IQR — повод проверить, а не основание автоматически удалять значение.

## Задание 4. Структура выручки

In [5]:
grand_total = total.sum()
assert grand_total != 0
share = total / grand_total * 100
assert np.isclose(share.sum(), 100)

table({"Регион": region_names, "Доля, %": share})
order = np.argsort(-share)
print("\nДоля крупнейшего региона:", region_names[order[0]], f"{share[order[0]]:.2f}%")
print("Доля двух крупнейших:", f"{share[order[:2]].sum():.2f}%")
print("Индекс Херфиндаля–Хиршмана (сумма квадратов долей, 0..1):", round(((share / 100) ** 2).sum(), 3), "(равные доли дали бы 0.2)")
print("Выручка сети без крупнейшего региона:", grand_total - total.max(), "тыс. (-", f"{share.max():.1f}%)")

Регион             Доля, %
Bishkek              23.96
Osh                  15.40
Karakol              29.94
Tokmok               11.27
Jalal-Abad           19.44

Доля крупнейшего региона: Karakol 29.94%
Доля двух крупнейших: 53.89%
Индекс Херфиндаля–Хиршмана (сумма квадратов долей, 0..1): 0.221 (равные доли дали бы 0.2)
Выручка сети без крупнейшего региона: 3103 тыс. (- 29.9%)


**Выводы.** Крупнейший регион — Karakol (**29,9%**), два крупнейших (Karakol и Bishkek) дают **53,9%** выручки. Индекс Херфиндаля–Хиршмана ≈ 0,22 при минимально возможном значении 0,20 для пяти регионов с равными долями — концентрация **умеренная**, критической зависимости от одного региона нет. Тем не менее потеря Karakol означала бы падение выручки сети на ~30%, поэтому нестабильность именно этого региона важна для бизнеса. С учётом выброса доля Karakol слегка завышена: без аномальной недели она составляет около 26% (см. задание 6).

## Задание 5. Сравнение периодов
Сравниваем среднее за недели 1–4 и 5–8 (срезы по оси недель).

In [6]:
first4 = sales[:, :4].mean(axis=1)
last4 = sales[:, 4:].mean(axis=1)
abs_change = last4 - first4

# Процентное изменение определяем только там, где базовое значение не равно нулю
safe_base = np.where(first4 != 0, first4, 1.0)
pct_change = np.where(first4 != 0, abs_change / safe_base * 100, np.nan)

table({"Регион": region_names, "Нед. 1-4": first4, "Нед. 5-8": last4, "Изм., абс.": abs_change, "Изм., %": pct_change})

net_pct = (sales[:, 4:].sum() / sales[:, :4].sum() - 1) * 100
print(f"\nРост по сети в целом: {net_pct:.2f}%")
print("Лучший рост:", region_names[np.argmax(pct_change)], "| худший рост:", region_names[np.argmin(pct_change)])

Регион            Нед. 1-4      Нед. 5-8    Изм., абс.       Изм., %
Bishkek             125.75        139.50         13.75         10.93
Osh                  81.50         89.00          7.50          9.20
Karakol             153.25        178.25         25.00         16.31
Tokmok               61.25         63.50          2.25          3.67
Jalal-Abad          103.75        111.50          7.75          7.47

Рост по сети в целом: 10.70%
Лучший рост: Karakol | худший рост: Tokmok


**Выводы.** Все регионы выросли. Сеть в целом — **+10,7%**. Bishkek (+10,9%) растёт вровень с сетью, Karakol показывает формально лучший рост (+16,3%), **но он завышен выбросом**: без недели 7 рост Karakol составляет +7,2% (см. задание 6). Osh (+9,2%) и Jalal-Abad (+7,5%) — умеренный рост, **Tokmok (+3,7%) — самая слабая динамика**, в 3 раза ниже среднего по сети. Базовые значения во всех регионах ненулевые, поэтому процентное изменение определено корректно (защита от деления на ноль через `np.where` в коде сохранена).

## Задание 6. Устойчивость результата
Исключаем выброс из расчёта с помощью маски и сравниваем суммы, средние и рейтинг. Дополнительно рассматриваем вариант с заменой выброса на медиану региона (чтобы сумма была сопоставима: при исключении теряется целая неделя).

In [7]:
keep = ~outlier_mask
cnt = keep.sum(axis=1)
total_clean = np.where(keep, sales, 0).sum(axis=1)
mean_clean = total_clean / cnt

# Вариант: выброс заменяем медианой региона (сопоставимая сумма за 8 недель)
total_imputed = np.where(outlier_mask, median[:, None], sales).sum(axis=1)

table({"Регион": region_names, "Сумма до": total, "Сумма после": total_clean, "Сумма (медиана)": total_imputed,
       "Среднее до": mean, "Среднее после": mean_clean}, dec=2)

rank = lambda x: np.argsort(np.argsort(-x)) + 1
print()
table({"Регион": region_names, "Ранг(сумма до)": rank(total), "Ранг(сумма после)": rank(total_clean),
       "Ранг(сред. до)": rank(mean), "Ранг(сред. после)": rank(mean_clean)}, dec=0)

print("\nОтрыв Karakol от Bishkek по сумме: до =", total[2] - total[0], "| после =", total_clean[2] - total_clean[0],
      "| при замене на медиану =", total_imputed[2] - total_imputed[0])

# Динамика Karakol без выброса
first4_c = np.where(keep[:, :4], sales[:, :4], np.nan)
last4_c = np.where(keep[:, 4:], sales[:, 4:], np.nan)
pct_clean = (np.nanmean(last4_c, axis=1) / np.nanmean(first4_c, axis=1) - 1) * 100
table({"Регион": region_names, "Изм.% до": pct_change, "Изм.% после": pct_clean})

cv_clean = np.sqrt(np.where(keep, (sales - mean_clean[:, None]) ** 2, 0).sum(axis=1) / (cnt - 1)) / mean_clean * 100
table({"Регион": region_names, "CV до, %": cv, "CV после, %": cv_clean})

Регион            Сумма до   Сумма после Сумма (медиана)    Среднее до Среднее после
Bishkek            1061.00       1061.00       1061.00        132.62        132.62
Osh                 682.00        682.00        682.00         85.25         85.25
Karakol            1326.00       1106.00       1265.00        165.75        158.00
Tokmok              499.00        499.00        499.00         62.38         62.38
Jalal-Abad          861.00        861.00        861.00        107.62        107.62

Регион       Ранг(сумма до) Ранг(сумма после) Ранг(сред. до) Ранг(сред. после)
Bishkek                  2             2             2             2
Osh                      4             4             4             4
Karakol                  1             1             1             1
Tokmok                   5             5             5             5
Jalal-Abad               3             3             3             3

Отрыв Karakol от Bishkek по сумме: до = 265 | после = 45 | при замене на м

**Выводы.**
- Исключение единственного выброса затрагивает только Karakol: сумма падает с 1326 до 1106, среднее — с 165,75 до 158, рост за период — с +16,3% до +7,2%, CV — с 13,9% до **5,0%** (после очистки Karakol оказывается на уровне стабильных регионов).
- **Рейтинг регионов не изменился** (Karakol, Bishkek, Jalal-Abad, Osh, Tokmok) ни по сумме, ни по среднему. Но **отрыв Karakol от Bishkek по сумме сократился с 265 до 45 тыс.** Это показывает, что при менее выраженном отрыве удаление (или, наоборот, сохранение) одного наблюдения вполне может поменять рейтинг. Сравнивать суммы после удаления корректно только осторожно: у Karakol остаётся 7 недель вместо 8, поэтому справедливее смотреть на среднее или на сумму с заменой выброса медианой (1265 против 1061 у Bishkek, т.е. Karakol по-прежнему лидер).
- Вывод об устойчивости: лидерство Karakol по объёму **устойчиво**, а его «рекордный рост» и «нестабильность» — результат одной недели и **неустойчивы**.

## Задание 7. Поиск проблемных регионов
**Критерий.** Регион считается проблемным, если выполняются **минимум два из трёх условий**:

| № | Показатель | Условие | Обоснование |
|---|---|---|---|
| F1 | Динамика (после очистки от выбросов) | рост между периодами меньше **половины роста сети** | регион стагнирует на фоне общего роста |
| F2 | Доля в выручке | доля между периодами **упала более чем на 0,5 п.п.** | регион теряет позиции относительно сети |
| F3 | Стабильность | CV по сырым данным **больше 10%** | продажи плохо прогнозируются |

Низкая абсолютная выручка сама по себе не критерий: размер региона учитывается через *динамику и изменение доли*, а не через итоговую сумму. Стабильный, но маленький регион, который растёт вместе с сетью, проблемным не считается.
Пороги (0,5 от роста сети, 0,5 п.п., 10%) — экспертные допущения, их стоит корректировать под бизнес-цели.

In [8]:
share_first = sales[:, :4].sum(axis=1) / sales[:, :4].sum() * 100
share_last = sales[:, 4:].sum(axis=1) / sales[:, 4:].sum() * 100
d_share = share_last - share_first

f1 = pct_clean < 0.5 * net_pct
f2 = d_share < -0.5
f3 = cv > 10
flags = f1.astype(int) + f2.astype(int) + f3.astype(int)
status = np.select([flags >= 2, flags == 1], ["ПРОБЛЕМНЫЙ", "наблюдение"], default="норма")

table({"Регион": region_names, "Рост очищ.,%": pct_clean, "Δ доли, п.п.": d_share, "CV, %": cv}, dec=2)
print()
table({"Регион": region_names, "F1": f1.astype(float), "F2": f2.astype(float), "F3": f3.astype(float), "Флагов": flags.astype(float)}, dec=0)
print("\nСтатус регионов:")
print(np.char.add(np.char.add(region_names, " -> "), status))

Регион        Рост очищ.,%  Δ доли, п.п.         CV, %
Bishkek              10.93          0.05          6.29
Osh                   9.20         -0.21          5.49
Karakol               7.23          1.48         13.93
Tokmok                3.67         -0.74          4.28
Jalal-Abad            7.47         -0.58          4.68

Регион                  F1            F2            F3        Флагов
Bishkek                  0             0             0             0
Osh                      0             0             0             0
Karakol                  0             0             1             1
Tokmok                   1             1             0             2
Jalal-Abad               0             1             0             1

Статус регионов:
['Bishkek -> норма' 'Osh -> норма' 'Karakol -> наблюдение' 'Tokmok -> ПРОБЛЕМНЫЙ' 'Jalal-Abad -> наблюдение']


**Результат.**
- **Tokmok — проблемный** (F1 и F2): рост всего +3,7% при сети +10,7%, доля в выручке снизилась с 11,7% до 10,9% (−0,74 п.п.). Продажи при этом очень стабильны — проблема не в разбросе, а в **стагнации**: регион отстаёт от остальных. Для окончательного вывода нужны данные о ёмкости рынка Tokmok (если он ограничен, то слабый рост может быть естественным).
- **Jalal-Abad — наблюдение** (F2): доля снизилась на 0,58 п.п., хотя рост +7,5% близок к норме.
- **Karakol — наблюдение** (F3): высокий CV вызван единственной неделей; после очистки CV = 5,0%, а рост +7,2% — выше порога F1. Это не проблемный регион, а регион с аномальным наблюдением.
- **Bishkek и Osh — норма.**
- Важно: Tokmok — самый маленький регион, но проблемным он признан не из-за малой выручки, а из-за отставания в динамике и потери доли.

## Задание 8. Аналитическая записка для руководства

**1. Какие регионы формируют основную выручку.**
Общая выручка сети — 4429 тыс. за 8 недель. Основу формируют **Karakol (29,9%) и Bishkek (24,0%)** — вместе 53,9%. Третий по вкладу — Jalal-Abad (19,4%), далее Osh (15,4%) и Tokmok (11,3%). Концентрация умеренная (индекс Херфиндаля–Хиршмана ≈ 0,22), критической зависимости от одного региона нет, но Karakol даёт почти треть выручки.

**2. Где наблюдаются нестабильные продажи.**
По сырым данным нестабилен **Karakol** (CV 13,9%), но причина — одна неделя (7-я: 220 тыс. при обычных 155–170). Без неё CV падает до 5,0%, то есть ядро продаж Karakol стабильно. Остальные регионы стабильны (CV 4,3–6,3%), самые ровные — Tokmok и Jalal-Abad. Отдельный риск — **Tokmok**: стабилен, но стагнирует (+3,7% при +10,7% по сети) и теряет долю.

**3. Какие отклонения требуют проверки.**
- Karakol, неделя 7 (220 тыс.): проверить по первичным данным — акция, крупный заказ или ошибка/дублирование. От ответа зависят и «рекордный рост» Karakol (+16,3% или +7,2%), и размер отрыва от Bishkek.
- Tokmok: причины слабой динамики; Jalal-Abad: небольшая потеря доли.
- Формально других выбросов по правилу IQR нет, но при 8 наблюдениях квартили оцениваются грубо, поэтому вывод о «чистоте» остальных регионов предварительный.

**4. Какие дополнительные данные нужны для объяснения причин.**
- данные по заказам Karakol за неделю 7 (количество, средний чек, крупнейшие заказы, акции, возвраты);
- календарь маркетинговых активностей и сезонность (праздники, акции по регионам);
- размер рынка/население/число активных клиентов по регионам (для оценки эффективности на душу и потенциала роста, особенно Tokmok);
- число заказов и средний чек отдельно (выручка = заказы × чек), каналы привлечения, доля повторных покупок;
- маржинальность и затраты по регионам (выручка ≠ прибыльность), логистика и уровень возвратов;
- более длинный ряд (≥ 6–12 месяцев), чтобы отличать тренд от случайных колебаний.

**Рекомендации.** Проверить выброс в Karakol; разобрать причины отставания Tokmok; оценивать регионы по динамике и доле рынка, а не по абсолютной выручке; не принимать решения по 8 неделям без дополнительных данных.

## Дополнительное задание. Сравнение ранжирований
Сравниваем ранги по общей выручке, средней выручке и стабильности (чем меньше CV, тем выше место). Для устойчивости берём CV после очистки от выброса.

In [9]:
rank_total = rank(total)
rank_mean = rank(mean)
rank_stab = np.argsort(np.argsort(cv)) + 1              # по сырому CV
rank_stab_clean = np.argsort(np.argsort(cv_clean)) + 1  # по очищенному CV
rank_growth = rank(pct_clean)

table({"Регион": region_names, "Сумма": rank_total.astype(float), "Среднее": rank_mean.astype(float),
       "Стабильность": rank_stab.astype(float), "Стаб.(очищ.)": rank_stab_clean.astype(float),
       "Рост(очищ.)": rank_growth.astype(float)}, dec=0)

Регион               Сумма       Среднее  Стабильность  Стаб.(очищ.)   Рост(очищ.)
Bishkek                  2             2             4             5             1
Osh                      4             4             3             4             2
Karakol                  1             1             5             3             4
Tokmok                   5             5             1             1             5
Jalal-Abad               3             3             2             2             3


**Выводы.**
- Ранги по общей и средней выручке полностью совпадают (периоды наблюдения одинаковы: среднее = сумма / 8). Это не два независимых показателя.
- Ранжирование по **стабильности** почти противоположно рейтингу по объёму: Karakol — 1-й по выручке, но последний по стабильности (по сырому CV), а Tokmok — последний по выручке, но самый стабильный. Причина: стабильность — относительная характеристика, а объём — абсолютная; к тому же более крупный регион при тех же относительных колебаниях даёт большие абсолютные отклонения.
- Даже после очистки от выброса Karakol (5-е место по сырому CV → 3-е по очищенному) не становится лидером по стабильности, а Bishkek, второй по объёму, — один из наименее стабильных (CV 6,3%). Лидер по объёму ≠ лидер по стабильности ≠ лидер по росту/эффективности.
- «Эффективность» нельзя оценить по одной выручке: нужны затраты, маржа и размер рынка. Региону с небольшой выручкой, но хорошей прибылью на единицу затрат или на душу населения, может быть эффективнее крупного.
- Поэтому для управленческих решений нужен набор метрик (объём, динамика, разброс, доля), а не один рейтинг.